# event_datasets.duckdb Exploration

This notebook helps you inspect and explore the DuckDB database at `./data/event_datasets.duckdb`.

In [1]:
from pathlib import Path
import duckdb
import polars as pl

pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_width_chars(140)

polars.config.Config

In [3]:
# Resolve the project root from the notebook location.
project_root = Path.cwd().resolve().parent
db_path = project_root / "data" / "event_datasets.duckdb"

print(f"Project root: {project_root}")
print(f"DB path: {db_path}")
print(f"Exists: {db_path.exists()}")

Project root: /Users/6161138/uv_projects/teamnl-events-dataset
DB path: /Users/6161138/uv_projects/teamnl-events-dataset/data/event_datasets.duckdb
Exists: True


In [4]:
con = duckdb.connect(str(db_path), read_only=True)
con.execute("SELECT current_database()").fetchone()

('event_datasets',)

## Discover Schema

In [5]:
# List tables and views in the database.
con.execute("SHOW ALL TABLES").pl()

database,schema,name,column_names,column_types,temporary
str,str,str,list[str],list[str],bool
"""event_datasets""","""main""","""kg_datasets""","[""dataset_id"", ""title"", … ""is_frequency_dataset""]","[""VARCHAR"", ""VARCHAR"", … ""BOOLEAN""]",false
"""event_datasets""","""main""","""kg_variables""","[""variable_id"", ""dataset_id"", … ""is_tijdstip""]","[""VARCHAR"", ""VARCHAR"", … ""BOOLEAN""]",false


In [6]:
# For each table, inspect schema and row count.
tables_df = con.execute("SHOW ALL TABLES").pl()
table_names = tables_df.get_column("name").drop_nulls().to_list() if "name" in tables_df.columns else []

summary = []
for table in table_names:
    count = con.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]
    summary.append({"table": table, "rows": count})

pl.DataFrame(summary).sort("rows", descending=True) if summary else pl.DataFrame()

table,rows
str,i64
"""kg_variables""",63136
"""kg_datasets""",578


In [8]:
# Pick one table to preview. Change selected_table as needed.
selected_table = table_names[1] if table_names else None
selected_table

'kg_variables'

In [9]:
if selected_table:
    display(con.execute(f"DESCRIBE \"{selected_table}\"").pl())
    display(con.execute(f"SELECT * FROM \"{selected_table}\" LIMIT 20").pl())
else:
    print("No tables found.")

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""variable_id""","""VARCHAR""","""YES""",null,null,null
"""dataset_id""","""VARCHAR""","""YES""",null,null,null
"""variable_name""","""VARCHAR""","""YES""",null,null,null
"""label_""","""VARCHAR""","""YES""",null,null,null
"""vocab_label""","""VARCHAR""","""YES""",null,null,null
…,…,…,…,…,…
"""definition""","""VARCHAR""","""YES""",null,null,null
"""num_predicates""","""VARCHAR""","""YES""",null,null,null
"""tijdstip_predicates""","""VARCHAR[]""","""YES""",null,null,null


variable_id,dataset_id,variable_name,label_,vocab_label,description,data_type,definition,num_predicates,tijdstip_predicates,valid_from,is_tijdstip
str,str,str,str,str,str,str,str,str,list[str],date,bool
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""MNDBEDRAGTWWW""","""Fiscaal maandbedrag TW-uitkeri…","""Fiscaal maandbedrag TW-uitkeri…","""Deze variabele wordt afgeleid …","""Integer""","""Fiscaal bedrag aan uitkeringen…","""15""",[],2017-01-01,false
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""SVOVRGHelpDipNiemand""","""Wie hebben je geholpen om te p…","""Wie hebben je geholpen om te p…",null,"""Integer""","""Heeft een specifiek persoon of…","""13""",[],2016-09-05,false
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""SILENQHB040""","""Day of household interview""","""Day of household interview""",null,"""Integer""","""Datum waarop een enquête is in…","""13""","[""https://portal.odissei.nl/schema/variableInformation#odisseiConceptVariableWaardestelselnaam""]",1989-01-01,true
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""SVOREGLWOONaGevOpl""","""Wel/niet leerwegondersteunendo…","""Geeft aan of men een indicatie…","""Met T wordt het jaar bedoeld w…","""Integer""","""Heeft een persoon een indicati…","""14""",[],2016-09-05,false
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""SVOVRGDiagnoseVsvGeenDeze""","""Van welke aandoeningen of ziek…","""Van welke aandoeningen of ziek…",null,"""Integer""","""Is voor een specifieke ziekte …","""13""",[],2014-01-01,false
…,…,…,…,…,…,…,…,…,…,…,…
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""Buit3Mnd""","""In buitenland tenminste 1 verb…","""In buitenland tenminste 1 verb…","""Was dit tenminste één keer voo…","""Integer""","""Het aantal keren dat een perso…","""14""",[],2017-05-01,false
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""RCBKEindGeldigheid""","""Datum einde geldigheid koppelr…","""Datum einde geldigheid koppelr…",null,"""String""","""Einddatum van de geldigheidspe…","""14""","[""https://portal.odissei.nl/schema/variableInformation#odisseiConceptVariableWaardestelselnaam""]",2006-01-01,true
"""https://kg.odissei.nl/.well-kn…","""https://doi.org/10.57934/0B01E…","""RINPERSOONS""","""codering voor de kwaliteit van…","""RinPersoons""",null,"""String""","""De bron waaruit een persoon id…","""13""",[],1904-12-31,false


## Example Analysis Queries

In [10]:
# Top 20 tables by row count (computed using metadata).
pl.DataFrame(summary).sort("rows", descending=True).head(20) if summary else pl.DataFrame()

table,rows
str,i64
"""kg_variables""",63136
"""kg_datasets""",578


In [11]:
# Close the connection when done.
con.close()